In [1]:
import pandas as pd 
import math # use for log function 
from collections import Counter 

In [2]:
file_path = "Processed_Reviews.csv" 
df = pd.read_csv(file_path) 

In [3]:
tokenized_reviews = df['tokenized'].dropna().apply(eval) 

In [4]:
def compute_tf(document): 
    word_count = Counter(document) 
    tf = {word: count / len(document) for word, count in word_count.items()} 
    return tf 

In [5]:
def compute_idf(documents): 
    N = len(documents)  # Total number of documents 
    idf = {} 
    all_words = set(word for doc in documents for word in doc)  # Unique words 
    for word in all_words: 
        count = sum(1 for doc in documents if word in doc) 
        idf[word] = math.log(N / count) 
    return idf 

In [6]:
def compute_tfidf(document, idf): 
    tfidf = {} 
    tf = compute_tf(document)  # Get TF values for the document 
    for word, tf_value in tf.items(): 
        tfidf[word] = tf_value * idf[word]  # Multiply TF and IDF 
    return tfidf 

In [7]:
documents = tokenized_reviews.tolist() 

In [8]:
tf_data = [compute_tf(doc) for doc in documents] 
tf_df = pd.DataFrame(tf_data).fillna(0) 
tf_df.to_csv("tf_scores.csv", index=False) 

In [9]:
idf = compute_idf(documents) 
idf_df = pd.DataFrame([idf]).fillna(0) 
idf_df.to_csv("idf_scores.csv", index=False) 

In [10]:
tfidf_data = [compute_tfidf(doc, idf) for doc in documents] 
tfidf_df = pd.DataFrame(tfidf_data).fillna(0) 
tfidf_df.to_csv("tfidf_scores.csv", index=False) 

In [ ]:
# Why we choose the HIGHEST scores (TF / IDF / TF-IDF):
# 1) Ranking purpose:
#    - These scores are *weights* that estimate how informative/important a term is.
#    - Higher weight = stronger signal that the term represents the document/topic.
#
# 2) TF (Term Frequency):
#    - TF measures how often a word appears in a document.
#    - Words with higher TF are more representative of that specific document’s content.
#    - So we pick the highest TF terms to get the main “keywords” of the document.
#
# 3) IDF (Inverse Document Frequency):
#    - IDF measures how rare a word is across all documents.
#    - Common words across many documents have low IDF (not useful to distinguish documents).
#    - Rare/specific words have high IDF (more discriminative).
#    - So we pick the highest IDF terms to find words that best differentiate documents.
#
# 4) TF-IDF (TF * IDF):
#    - TF-IDF combines:
#        (a) importance inside the document (TF)
#        (b) uniqueness across the dataset (IDF)
#    - A high TF-IDF term usually means:
#        - it appears frequently in the document
#        - BUT not in many other documents
#    - So we pick the highest TF-IDF terms because they are the best “top keywords”
#      that summarize the document and help distinguish it from others.
#
# 5) Practical reason (output readability):
#    - Keeping only the top/highest scores reduces noise and avoids listing many
#      low-value terms that contribute little meaning.
#    - This makes the results easier to interpret and use (e.g., for keyword extraction,
#      summarization, search, or classification features).